# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/hands-on-multimodal-AI/blob/main/hands-on/HANDS_ON_session_04_AudioFlamingo3_meeting_analysis.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://www.oreilly.com/library/view/transformers-the-definitive/9781098167004/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="Transformers: The Definitive Guide"/>
</a>




<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# Hands-on: Meeting Analysis with Audio Flamingo 3

In this hands-on you will use **[Audio Flamingo 3 (AF3)](https://huggingface.co/nvidia/audio-flamingo-3-hf)** -- NVIDIA's instruction-following Large Audio-Language Model -- to analyze a multi-speaker meeting recording.

Unlike traditional pipelines that chain separate models (diarization -> ASR -> text LLM), AF3 understands audio **natively** and can:

| Capability | Traditional Pipeline | AF3 (this notebook) |
|---|---|---|
| Transcription | Dedicated ASR model | Built-in |
| Speaker tone / sentiment | Not available | Understands paralinguistics |
| Summarization | Separate text LLM on transcript | Directly from audio |
| Q&A | Transcript -> text LLM | Directly from audio |
| Multi-turn dialogue | Not supported | Conversational |

### What You Will Do

1. **Load & listen** to a multi-speaker audio clip
2. **Transcribe** the meeting with AF3
3. **Analyze sentiment & tone** -- go beyond text to understand *how* things are said
4. **Summarize** the meeting directly from audio
5. **Multi-turn Q&A** -- have a conversation about the audio
6. **Per-speaker segment analysis** -- combine pyannote diarization with AF3 for segment-level insights

> **Note:** AF3 processes audio in 30-second windows with a 10-minute total cap per sample. This model is for non-commercial research purposes only. If you want truly open source and intruction following model, I recommend using [Kimi Audio](https://github.com/MoonshotAI/Kimi-Audio), just note you need flash attention 2, which causes problems in Colab environments sometimes, and is the reason I don't use it here.

### Prerequisites

- A Hugging Face account and token (free)
- Accept the model terms for:
  - [nvidia/audio-flamingo-3-hf](https://huggingface.co/nvidia/audio-flamingo-3-hf)
  - [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) (for the bonus section)
  - [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) (for the bonus section)

## Setup & Installation

In [ ]:
!pip install --upgrade git+https://github.com/huggingface/transformers accelerate datasets soundfile pyannote.audio numpy -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Restart the runtime after installing packages above.
# In Colab: Runtime -> Restart session, then continue from the next cell.
# Uncomment the line below to auto-restart:

# import os; os.kill(os.getpid(), 9)

In [ ]:
# Log in to Hugging Face (needed for gated models like AF3 and pyannote)
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [ ]:
import torch
import numpy as np
import soundfile as sf
from datasets import load_dataset
from IPython.display import Audio, display
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

## Load the Meeting Audio

We use a multi-speaker LibriSpeech dataset -- a concatenation of multiple speakers reading different passages. This simulates a meeting with several participants.

In [ ]:
# Load multi-speaker audio dataset
dataset = load_dataset("sanchit-gandhi/concatenated_librispeech", split="train")
sample = dataset[0]

audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]

print(f"Audio duration: {len(audio_array) / sr:.1f} seconds")
print(f"Sample rate:    {sr} Hz")

# Save to a temp WAV file (AF3 needs a file path)
AUDIO_PATH = "/tmp/meeting_audio.wav"
sf.write(AUDIO_PATH, audio_array, sr)
print(f"Saved to:       {AUDIO_PATH}")

# Play the full audio
print("\nFull meeting audio:")
display(Audio(audio_array, rate=sr))

README.md:   0%|          | 0.00/359 [00:00<?, ?B/s]

data/train-00000-of-00001-4f2454a2c146e6(…):   0%|          | 0.00/661k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Audio duration: 21.7 seconds
Sample rate:    16000 Hz
Saved to:       /tmp/meeting_audio.wav

Full meeting audio:


## Load Audio Flamingo 3

Load the AF3 model and processor. You'll use `bfloat16` precision and `device_map="auto"` to fit in GPU memory.

In [ ]:
model_id = "nvidia/audio-flamingo-3-hf"

processor = AutoProcessor.from_pretrained(model_id)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"AF3 model loaded: {model_id}")

processor_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/703 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/788 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

AF3 model loaded: nvidia/audio-flamingo-3-hf


In [ ]:
# Helper function to query AF3
def ask_af3(prompt, audio_path=None, conversation=None, max_new_tokens=512):
    """
    Send a prompt (with optional audio) to AF3 and return the response.
    Supports multi-turn by passing the conversation list back in.
    """
    if conversation is None:
        content = [{"type": "text", "text": prompt}]
        if audio_path:
            content.append({"type": "audio", "path": audio_path})
        conversation = [{"role": "user", "content": content}]
    else:
        # Multi-turn: append new user message (text-only follow-up)
        conversation.append({
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        })

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)

    # Cast float tensors to model dtype (audio features are float32, model is bfloat16)
    for key in inputs:
        if inputs[key].is_floating_point():
            inputs[key] = inputs[key].to(model.dtype)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    response = processor.batch_decode(
        outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

    # Append assistant response for multi-turn
    conversation.append({"role": "assistant", "content": [{"type": "text", "text": response}]})

    return response, conversation

print("ask_af3() helper ready.")

ask_af3() helper ready.


---

## Transcribe the Meeting

Ask AF3 to transcribe the audio. Unlike a pure ASR model, AF3 is an **instruction-following** model -- you tell it what you want in natural language.

> **Design note:** Sections 4–7 are chained as a **multi-turn conversation** (`conv`). This way AF3 builds context turn by turn and correctly identifies all speakers from the start.

In [ ]:
# Start the multi-turn conversation with the audio
transcription, conv = ask_af3(
    "How many different speakers can you identify in this audio? "
    "Transcribe the input speech and label each speaker.",
    audio_path=AUDIO_PATH,
    max_new_tokens=1024,
)

print("--- Transcription ---\n")
print(transcription)

--- Transcription ---

There are two speakers in this audio. The first speaker is a male with a neutral tone, and the second speaker is a female with a neutral tone.


---

## Analyze Speaker Sentiment & Tone

This is where AF3 shines compared to traditional ASR. It does not just convert speech to text -- it **understands how things are said**. Let's ask about the emotional tone of the speakers.

In [ ]:
# Continue the conversation -- AF3 already knows the speakers from the transcription turn
sentiment, conv = ask_af3(
    (
        "Now analyze the tone, sentiment, and speaking style of each speaker you identified. "
        "For each speaker, describe:\n"
        "- Their emotional tone (e.g., calm, excited, nervous, confident)\n"
        "- Their speaking pace and energy level\n"
        "- Any notable changes in sentiment throughout their speech"
    ),
    conversation=conv,
    max_new_tokens=1024,
)

print("--- Sentiment & Tone Analysis ---\n")
print(sentiment)

--- Sentiment & Tone Analysis ---

The male speaker has a neutral tone, speaking at a moderate pace with a calm energy level. The female speaker also has a neutral tone, speaking at a moderate pace with a calm energy level.


---

## Summarize the Meeting

Ask AF3 to generate a concise summary **directly from the audio** -- no text transcript needed as an intermediate step.

In [ ]:
# Continue the conversation
summary, conv = ask_af3(
    (
        "Based on everything you've heard, provide a concise summary of this recording. "
        "What are the main topics discussed and what are the key points from each speaker?"
    ),
    conversation=conv,
    max_new_tokens=512,
)

print("--- Meeting Summary ---\n")
print(summary)

--- Meeting Summary ---

The male speaker discusses the concept of sovereignty and its exercise in France, while the female speaker discusses the impact of a wife's actions on a man's future.


---

## Multi-turn Q&A about the Audio

Continue the conversation with deeper follow-up questions. The model remembers everything from the transcription, sentiment, and summary turns above.

In [ ]:
# Continue from the same conversation
answer1, conv = ask_af3(
    "Which speaker sounds the most confident or enthusiastic? Why do you think so?",
    conversation=conv,
)

print("Q: Which speaker sounds most confident/enthusiastic?\n")
print(f"A: {answer1}")

Q: Which speaker sounds most confident/enthusiastic?

A: The male speaker sounds the most confident, as his tone is steady and assured, and he speaks at a moderate pace with a calm energy level.


In [ ]:
# Follow-up
answer2, conv = ask_af3(
    "Do any of the speakers sound uncertain, nervous, or hesitant? What makes you think so?",
    conversation=conv,
)

print("Q: Any speakers sound uncertain or hesitant?\n")
print(f"A: {answer2}")

Q: Any speakers sound uncertain or hesitant?

A: The female speaker sounds slightly hesitant at times, as her tone is less assured and she speaks at a slower pace with a lower energy level.


In [ ]:
# Another follow-up
answer3, conv = ask_af3(
    "If this were a real meeting, what action items or next steps would you suggest based on the discussion?",
    conversation=conv,
)

print("Q: What action items would you suggest?\n")
print(f"A: {answer3}")

Q: What action items would you suggest?

A: Based on the discussion, it seems that the male speaker is proposing a new law or policy, and the female speaker is expressing concern about the potential consequences of a wife's actions. The next steps could involve further discussion, consultation with experts, or taking action to address the issues raised.


---

## Per-Speaker Segment Analysis

Combine **pyannote.audio** for speaker diarization (who speaks when) with **AF3** for rich per-segment analysis. This lets you:
- Play each speaker's audio individually
- Get AF3's analysis of each segment's content *and* tone

> **Note:** Accept the pyannote model terms first:
> - [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1)
> - [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0)

In [ ]:
from pyannote.audio import Pipeline

# Load diarization pipeline
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=True,
)
diarization_pipeline = diarization_pipeline.to(torch.device("cuda:0"))

# Run diarization
waveform = torch.tensor(audio_array[None, :]).float()
diarization = diarization_pipeline({"waveform": waveform, "sample_rate": sr})

# Extract speaker segments (compatible with pyannote v3 and v4+)
MIN_SEGMENT_SAMPLES = 400  # skip segments shorter than ~25 ms

raw_segments = []
if hasattr(diarization, "speaker_diarization"):
    for turn, speaker in diarization.speaker_diarization:
        raw_segments.append((turn.start, turn.end, speaker))
else:
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        raw_segments.append((turn.start, turn.end, speaker))

speaker_segments = []
for start, end, speaker in raw_segments:
    s, e = int(start * sr), int(end * sr)
    seg_audio = audio_array[s:e]
    if len(seg_audio) < MIN_SEGMENT_SAMPLES:
        continue
    speaker_segments.append({"speaker": speaker, "start": start, "end": end, "audio": seg_audio})

print(f"Found {len(speaker_segments)} speaker segments:")
for i, seg in enumerate(speaker_segments):
    print(f"  [{i}] {seg['speaker']}  {seg['start']:.1f}s - {seg['end']:.1f}s  ({seg['end'] - seg['start']:.1f}s)")

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


Found 3 speaker segments:
  [0] SPEAKER_01  0.7s - 7.2s  (6.5s)
  [1] SPEAKER_01  7.7s - 14.5s  (6.8s)
  [2] SPEAKER_00  15.4s - 21.5s  (6.1s)


In [ ]:
# Analyze each speaker segment with AF3
for i, seg in enumerate(speaker_segments):
    # Save segment to temp file
    seg_path = f"/tmp/segment_{i}.wav"
    sf.write(seg_path, seg["audio"], sr)

    # Play the segment
    print(f"\n{'=' * 60}")
    print(f"Segment {i} -- {seg['speaker']} ({seg['start']:.1f}s - {seg['end']:.1f}s)")
    display(Audio(seg["audio"], rate=sr))

    # Ask AF3 to transcribe and analyze this segment
    analysis, _ = ask_af3(
        (
            "For this audio segment:\n"
            "1. Transcribe exactly what is said\n"
            "2. Describe the speaker's tone and emotional state\n"
            "3. Rate their confidence level (low / medium / high)"
        ),
        audio_path=seg_path,
        max_new_tokens=300,
    )

    print(f"\n{analysis}")


Segment 0 -- SPEAKER_01 (0.7s - 7.2s)



1. The second in importance is as follows sovereignty may be defined to be the right of making laws.

Segment 1 -- SPEAKER_01 (7.7s - 14.5s)



1. In France the king really exercises a portion of the sovereign power since the laws have no weight

Segment 2 -- SPEAKER_00 (15.4s - 21.5s)



1. HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE.


---

## Try Your Own Questions!

Change the prompt below to ask anything about the meeting audio. Some ideas to try:

- *"What is the overall mood of this recording?"*
- *"Is there any disagreement between the speakers?"*
- *"Generate a detailed caption describing all speech, sounds, and background noise."*
- *"What language are the speakers using? Any accents?"*
- *"If this were a real meeting, what action items would you suggest?"*

In [ ]:
my_question = "What is the overall mood of this recording? Are the speakers engaged or monotone?"

answer, _ = ask_af3(my_question, audio_path=AUDIO_PATH)

print(f"Q: {my_question}\n")
print(f"A: {answer}")

Q: What is the overall mood of this recording? Are the speakers engaged or monotone?

A: The overall mood of this recording is serious and informative. The speakers are engaged and not monotone.


In [ ]:
my_question = "Generate a detailed caption for the input audio, describing all notable speech, sound, and background events comprehensively."

answer, _ = ask_af3(my_question, audio_path=AUDIO_PATH, max_new_tokens=1024)

print(f"Q: {my_question}\n")
print(f"A: {answer}")

Q: Generate a detailed caption for the input audio, describing all notable speech, sound, and background events comprehensively.

A: A male speaker, with a neutral tone and moderate pace, discusses the concept of sovereignty and its implications in France, while the background features the sound of a book being read aloud.
